# EXERCISE SOLUTION

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

In [2]:
!ollama pull llama3.2:1b

pulling manifest â ‹ pulling manifest â ™ pulling manifest â ¹ pulling manifest â ¸ pulling manifest â ¼ pulling manifest â ´ pulling manifest â ¦ pulling manifest â § pulling manifest â ‡ pulling manifest â � pulling manifest â ‹ pulling manifest 
pulling 74701a8c35f6: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 1.3 GB                         
pulling 966de95ca8a6: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 1.4 KB                         
pulling fcc5a6bec9da: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 7.7 KB                         
pulling a70ff7e570d9: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 6.0 KB                         
pulling 4f659a1e86d7: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�  485 B                         
verifying sha256 digest 
writing manifest 
success 


In [4]:
!pip install ollama

  Using cached ollama-0.6.1-py3-none-any.whl.metadata (4.3 kB)
Using cached ollama-0.6.1-py3-none-any.whl (14 kB)


In [5]:
# imports

import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import ollama

In [6]:
# Constants

MODEL = "llama3.2:1b"

In [7]:
# A class to represent a Webpage

class Website:
    """
    A utility class to represent a Website that we have scraped
    """
    url: str
    title: str
    text: str

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [8]:
# Let's try one out

ed = Website("https://edwarddonner.com")
print(ed.title)
print(ed.text)

Home - Edward Donner
Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 400,000 enrolled across 190 

## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT4o have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [9]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

In [10]:
# A function that writes a User Prompt that asks for summaries of websites:

def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "The contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

## Messages

The API from Ollama expects the same message format as OpenAI:

```
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]

In [11]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

## Time to bring it together - now with Ollama instead of OpenAI

In [14]:
# And now: call the Ollama function instead of OpenAI

def summarize(url):
    website = Website(url)
    messages = messages_for(website)
    response = ollama.chat(model=MODEL, messages=messages)
    return response['message']['content']

In [13]:
summarize("https://edwarddonner.com")

'### Home Page Summary\n\n* The website is about the author, Ed, who is a proficient AI engineer and entrepreneur.\n* He mentions his background in AI startups (untapt) and his current role as CTO at Nebula.io.\n\n### Posts\n\n#### Recent Post\n\n* "Welcome to my channel! I\'m excited to share my journey with you all."\n\t+ No additional context or summary provided.\n\n#### Upcoming Announcement\n\n* "AI Builder with n8n - Create Agents and Voice Agents"\n\t+ No additional information provided about the announcement.\n\n### About Page Summary\n\n* The author is Ed, a proficient AI engineer and entrepreneur.\n* He mentions his background in AI startups (untapt) and his current role as CTO at Nebula.io.\n* He also mentions being an amateur electronic music producer and enjoying Hacker News.\n\n### Posts\n\n#### Past Post\n\n* "When you finally figure out how to use an LLM, but it still doesn\'t understand sarcasm"\n\t+ No additional context or summary provided.\n\n#### Future Announcemen

In [15]:
# A function to display this nicely in the Jupyter output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [16]:
display_summary("https://edwarddonner.com")

### Home
The website appears to be a personal blog and profile created by Edward Donner, who is an AI expert and entrepreneur. The content showcases his work on LLMs (Large Language Models), his involvement with various projects such as Nebula.io and untapt, and his passion for coding, electronic music production, and sharing knowledge.

### Posts
The website includes several posts from Edward Donner's perspective, including:

* A brief introduction to himself and his interests in AI and LLMs.
* Updates on his work, including new projects and courses he has created (e.g., Udemy courses).
* Personal anecdotes about his experiences with AI, such as his success in creating best-selling courses.

### Home
The website also includes a section titled "Home", which appears to be an introduction to the website itself. It seems that Edward Donner is enthusiastic about sharing his knowledge and experiences with others who are interested in AI.

### AI Curriculum
The "AI Curriculum" section is not explicitly stated, but it can be inferred from the context of other posts and projects mentioned on the website. However, a summary of what this might entail is:

* A collection of courses or resources that Edward Donner has created to teach others about AI and its applications.
* Possibly tutorials, guides, or workshops he has offered in the past.

### Proficient AI Engineer
There is no explicit information on this topic, but it can be inferred from other posts and projects mentioned on the website. A summary of what this might entail is:

* A description of Edward Donner's experience as an experienced AI engineer.
* Possibly a list of his accomplishments or certifications in this field.

### Connect Four
There is no explicit information about "Connect Four" on the website, but it can be inferred from other posts and projects mentioned. A summary of what this might entail is:

* An explanation of Edward Donner's involvement with this project.
* Possibly a description of how he contributes to or leads this initiative.

### Outsmart
The "Outsmart" section appears to be an interactive project where LLMs compete against each other in a battle of diplomacy and deviousness. A summary of what this might entail is:

* An explanation of the concept behind this project.
* Possibly information on how users can participate or engage with this project.

### About
The "About" section provides basic information about Edward Donner, including his name, interests, and professional affiliations. It also includes a contact email address for those interested in reaching out to him.

### Posts
The website includes several posts from Edward Donner's perspective, which can be summarized as follows:

* A brief introduction to himself and his work on LLMs.
* Updates on his projects, including Nebula.io and untapt.
* Personal anecdotes about his experiences with AI and sharing knowledge with others.

# Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach.

Websites that are rendered with Javascript, like React apps, won't show up. See the community-contributions folder for a Selenium implementation that gets around this. You'll need to read up on installing Selenium (ask ChatGPT!)

Also Websites protected with CloudFront (and similar) may give 403 errors - many thanks Andy J for pointing this out.

But many websites will work just fine!

In [17]:
display_summary("https://cnn.com")

**Summary:**

Breaking News, Latest News and Videos is a website that aggregates news from various sources, including CNN. The website provides summaries of articles, as well as information about recent news stories.

**News Articles:**

* Ukraine-Russia War: A recent report by CNN states that the US and Iran have concluded high-stakes talks in Oman, with Trump not apologizing for sharing a racist video depicting the Obamas as apes.
* Israel-Hamas War: CNN reports on the latest developments in the conflict between Israel and Hamas, including a message from Olympic athletes from Greenland expressing solidarity.
* Winter Olympics 2026: The opening ceremony of the Winter Olympics has been viewed by millions online, with highlights of the event available on CNN's website.

**Ad Feedback:**

* Ad froze or did not finish loading
* Video content never loaded
* Audio on ad was too loud
* Ad never loaded
* Ad prevented/slowed the page from loading

Overall, the website appears to be experiencing technical issues with its video player and ad functionality.

In [18]:
display_summary("https://anthropic.com")

# Anthropic Overview

Anthropic is a public benefit corporation dedicated to securing its benefits and mitigating its risks. The company aims to have AI impact on the world, with a focus on safety and responsible development.

# Research and Development

Anthropic focuses on building AI for humanity's long-term well-being. They are developing models like Claude Opus 4.6, which is the most powerful model for coding, agents, and professional work. Their Responsible Scaling Policy addresses AI safety concerns.

# Economic Impact

Anthropic is part of a global economic index that tracks various aspects of economies around the world. The company's economic future is closely tied to its ability to provide beneficial AI solutions.

# Education and Training

Anthropic offers various resources for education, including tutorials, developer documentation, and courses on their platform. They also partner with startups, governments, healthcare organizations, and nonprofits to develop AI solutions that benefit society.

# Security and Compliance

Anthropic prioritizes security and compliance, ensuring that their products meet the highest standards of safety and responsibility. They also provide resources for developers to learn about their platform and follow best practices in responsible development.

# Products and Services

Anthropic offers several products and services, including Claude, a space for genuine conversations without ads or sponsored content. Their developer platform is available on multiple platforms, including Amazon Bedrock, Google Cloud's Vertex AI, and Slack.

# Sharing your code

I'd love it if you share your code afterwards so I can share it with others! You'll notice that some students have already made changes (including a Selenium implementation) which you will find in the community-contributions folder. If you'd like add your changes to that folder, submit a Pull Request with your new versions in that folder and I'll merge your changes.

If you're not an expert with git (and I am not!) then GPT has given some nice instructions on how to submit a Pull Request. It's a bit of an involved process, but once you've done it once it's pretty clear. As a pro-tip: it's best if you clear the outputs of your Jupyter notebooks (Edit >> Clean outputs of all cells, and then Save) for clean notebooks.

PR instructions courtesy of an AI friend: https://chatgpt.com/share/670145d5-e8a8-8012-8f93-39ee4e248b4c